In [ ]:
import os
# from google.colab import userdata
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

RESOURCE_GROUP = "cis-5270-team-10"
OPENAI_API_KEY = "put your api key here"

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = "put your subscription id here"

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = "CIS-5270"
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT

CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

BASE_MODEL = "gpt-4.1-nano-2025-04-14"

BASE_DEPLOYMENT = "gpt-4.1-nano"

print("Connected to Azure OpenAI")

Connected to Azure OpenAI


# Direct Preference Optimization

Fine-tune GPT-4.1-nano on preference pairs of the form:

- prompt = FEVER passage + claim
- chosen = better answer (correct label + grounded justification)
- rejected = worse answer (wrong label or flawed justification)

1. converting the DPO data into Azure format
2. uploading the training file
3. submitting the DPO fine-tuning job
4. monitoring job status
5. preparing comparable dev-set evaluation
6. stopping before deployment

## Step 1: Load DPO Training Data

We start from `dpo_data_3k.jsonl`, which already contains the final preference pairs:

- `prompt`
- `chosen`
- `rejected`

Unlike SFT, DPO does not use standard `{"messages": [...]}` training rows.
Instead, Azure expects each row to contain:

- `input`
- `preferred_output`
- `non_preferred_output`

Here, we convert the dataset into that format below.

In [5]:
#imports, seed, helpers
import json
import os
import time
import random
import re
from collections import Counter, defaultdict
from tqdm import tqdm

random.seed(42)
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
data_dir = "data/generated"


dpo_path = f"{data_dir}/dpo_data_3k.jsonl"
dev_path = "data/joined/fever_dev_joined.jsonl"

print("DPO path exists:", os.path.exists(dpo_path))
print("DEV path exists:", os.path.exists(dev_path))

dpo_data = load_jsonl(dpo_path)
dev_data = load_jsonl(dev_path)

print("DPO rows:", len(dpo_data))
print("Dev rows:", len(dev_data))
print("DPO example keys:", dpo_data[0].keys())
print("Dev example keys:", dev_data[0].keys())

DPO path exists: True
DEV path exists: True
DPO rows: 3000
Dev rows: 19891
DPO example keys: dict_keys(['id', 'prompt', 'chosen', 'rejected'])
Dev example keys: dict_keys(['claim', 'passage', 'label', 'id'])


## Step 2: Define the Matched Prompt Format

To keep DPO evaluation comparable to the baseline and SFT notebooks,
we use the same fact-checking format:

- output one label
- then one sentence justification
- labels must be exactly:
  - SUPPORTED
  - CONTRADICTED
  - NOT MENTIONED

We keep this instruction in the DPO training examples so the learned behavior matches our evaluation setup.

In [ ]:
PROMPT_MATCHED_SYSTEM = """You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.
Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"""

## The EF prompt for this experiment

In [ ]:
PROMPT_MATCHED_SYSTEM_2 = """You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a justification directly quoting the relavent part of the passage. Make sure you quote the part of the passage exactly unless it is not mentioned.
Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"""

## Step 3: Convert DPO Data into Azure DPO Format

Each row is converted into:

- `input.messages`
- `preferred_output`
- `non_preferred_output`

We also remove the original `id` field and reshapes the example into the schema Azure expects for DPO.

In [19]:
def convert_row_to_dpo_format(row, system_prompt):
    return {
        "input": {
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": row["prompt"]
                }
            ]
        },
        "preferred_output": [
            {
                "role": "assistant",
                "content": row["chosen"]
            }
        ],
        "non_preferred_output": [
            {
                "role": "assistant",
                "content": row["rejected"]
            }
        ]
    }

converted_rows = [convert_row_to_dpo_format(row, PROMPT_MATCHED_SYSTEM) for row in dpo_data]

print("Converted rows:", len(converted_rows))
print(json.dumps(converted_rows[0], indent=2)[:1500])

Converted rows: 3000
{
  "input": {
    "messages": [
      {
        "role": "system",
        "content": "You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.\nFormat: LABEL: justification sentence\nLabel must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"
      },
      {
        "role": "user",
        "content": "Passage: The law of value -LRB- German : Wertgesetz -RRB- is a central concept in Karl Marx 's critique of political economy , first expounded in his polemic The Poverty of Philosophy -LRB- 1847 -RRB- against Pierre-Joseph Proudhon , with reference to David Ricardo 's economics.See Marx , The Poverty of Philosophy , chapter 1 part 2 where Marx refers to Proudhon 's own `` law of value '' and chapter 3 , titled `` Application of the Law of the Proportionality of Value '' .\n\nClaim: Law of value is a peripheral concept in Karl Marx's critique of pol

In [20]:
converted_rows_2 = [convert_row_to_dpo_format(row, PROMPT_MATCHED_SYSTEM_2) for row in dpo_data]

print("Converted rows:", len(converted_rows_2))
print(json.dumps(converted_rows_2[0], indent=2)[:1500])

Converted rows: 3000
{
  "input": {
    "messages": [
      {
        "role": "system",
        "content": "You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a justification directly quoting the relavent part of the passage. Make sure you quote the part of the passage exactly unless it is not mentioned.\nFormat: LABEL: justification sentence\nLabel must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"
      },
      {
        "role": "user",
        "content": "Passage: The law of value -LRB- German : Wertgesetz -RRB- is a central concept in Karl Marx 's critique of political economy , first expounded in his polemic The Poverty of Philosophy -LRB- 1847 -RRB- against Pierre-Joseph Proudhon , with reference to David Ricardo 's economics.See Marx , The Poverty of Philosophy , chapter 1 part 2 where Marx refers to Proudhon 's own `` law of value '' and chapter 3 , titled `` Application of the Law of the Proportionality of Value '' .\n\nC

## Step 4: Basic Local Cleaning

Remove rows that are obviously unsafe for training:

- missing messages
- empty strings
- chosen = rejected

Sanity check before Azure preprocessing.

In [10]:
cleaned_rows = []

for row in converted_rows:
    try:
        system_msg = row["input"]["messages"][0]["content"].strip()
        user_msg = row["input"]["messages"][1]["content"].strip()
        chosen = row["preferred_output"][0]["content"].strip()
        rejected = row["non_preferred_output"][0]["content"].strip()
    except Exception:
        continue

    if not system_msg or not user_msg or not chosen or not rejected:
        continue
    if chosen == rejected:
        continue

    cleaned_rows.append(row)

print("Usable DPO rows:", len(cleaned_rows))
print("Dropped rows:", len(converted_rows) - len(cleaned_rows))

Usable DPO rows: 3000
Dropped rows: 0


## EF section

The variables that has _2 postfix are for EF

In [25]:
cleaned_rows_2 = []

for row in converted_rows_2:
    try:
        system_msg = row["input"]["messages"][0]["content"].strip()
        user_msg = row["input"]["messages"][1]["content"].strip()
        chosen = row["preferred_output"][0]["content"].strip()
        rejected = row["non_preferred_output"][0]["content"].strip()
    except Exception:
        continue

    if not system_msg or not user_msg or not chosen or not rejected:
        continue
    if chosen == rejected:
        continue

    cleaned_rows_2.append(row)

print("Usable DPO rows:", len(cleaned_rows_2))
print("Dropped rows:", len(converted_rows_2) - len(cleaned_rows_2))

Usable DPO rows: 3000
Dropped rows: 0


## Step 5: Save Azure Upload File

Unlike SFT, we are not using a validation split or hyperparameter sweep here.
We train on the full cleaned DPO dataset and evaluate later on a separate held-out FEVER dev set.

In [ ]:
save_jsonl("dpo_train_azure.jsonl", cleaned_rows)
print("Saved dpo_train_azure.jsonl with", len(cleaned_rows), "rows")

Saved dpo_train_azure.jsonl with 3000 rows


In [ ]:
save_jsonl("dpo_train_azure_2.jsonl", cleaned_rows_2)
print("Saved dpo_train_azure_2.jsonl with", len(cleaned_rows_2), "rows")

Saved dpo_train_azure_2.jsonl with 3000 rows


## Step 6: Upload the Training File

Azure fine-tuning requires the training file to be uploaded first.
When uploaded, we have Azure assign a file ID that is used when creating the DPO job.

In [12]:
print("Uploading DPO training file...")
with open("dpo_train_azure.jsonl", "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")

train_file_id = train_file.id
print("Training file ID:", train_file_id)

Uploading DPO training file...
Training file ID: file-7ba075ac1696455e91572e46b3d1a71e


In [27]:
print("Uploading DPO training file...")
with open("dpo_train_azure_2.jsonl", "rb") as f:
    train_file_2 = openai_client.files.create(file=f, purpose="fine-tune")

train_file_id_2 = train_file_2.id
print("Training file ID:", train_file_id_2)

Uploading DPO training file...
Training file ID: file-8750b464e24e410eb6e1815d81860a24


## Step 7: Submit the DPO Fine-Tuning Job

We use one DPO configuration:

- lr multiplier = 1.0
- epochs = 1
- batch size = 1

This mirrors the “simple baseline config” style used in the SFT notebook.

In [ ]:
print(f"Creating DPO fine-tuning job for {BASE_MODEL}")

job = openai_client.fine_tuning.jobs.create(
    training_file=train_file_id,
    model=BASE_MODEL,
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {
                "n_epochs": 1,
                "batch_size": 1,
                "learning_rate_multiplier": 1.0
            }
        }
    },
    extra_body={"trainingType": "GlobalStandard"},
    suffix="dpo_main"
)

print("Job ID:", job.id)
print("Initial status:", job.status)

Creating DPO fine-tuning job for gpt-4.1-nano-2025-04-14
Job ID: ftjob-58abe7a24f934eb68e5040dbb8010565
Initial status: pending


In [28]:
print(f"Creating DPO fine-tuning job for {BASE_MODEL}")

job_2 = openai_client.fine_tuning.jobs.create(
    training_file=train_file_id_2,
    model=BASE_MODEL,
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {
                "n_epochs": 1,
                "batch_size": 1,
                "learning_rate_multiplier": 1.0
            }
        }
    },
    extra_body={"trainingType": "GlobalStandard"},
    suffix="dpo_main"
)

print("Job ID:", job_2.id)
print("Initial status:", job_2.status)

Creating DPO fine-tuning job for gpt-4.1-nano-2025-04-14
Job ID: ftjob-811fb4a2a03c40ac82027dd09fd46eed
Initial status: pending


## Step 8: Monitor the DPO Job

In [30]:
def wait_for_job(job_id, poll_seconds=30):
    while True:
        current = openai_client.fine_tuning.jobs.retrieve(job_id)
        print(f"{job_id}: {current.status}")

        if current.status in {"succeeded", "failed", "cancelled"}:
            return current

        time.sleep(poll_seconds)



In [ ]:
final_job = wait_for_job(job.id, poll_seconds=30)

In [54]:
final_job_2 = wait_for_job(job_2.id, poll_seconds=30)

ftjob-811fb4a2a03c40ac82027dd09fd46eed: succeeded


In [55]:
current = openai_client.fine_tuning.jobs.retrieve(job_2.id)
print(f"{job_2.id}: {current.status}")
print(current)

ftjob-811fb4a2a03c40ac82027dd09fd46eed: succeeded
FineTuningJob(id='ftjob-811fb4a2a03c40ac82027dd09fd46eed', created_at=1777165251, error=None, fine_tuned_model='gpt-4.1-nano-2025-04-14.ft-811fb4a2a03c40ac82027dd09fd46eed-dpo_main', finished_at=1777214737, hyperparameters=None, model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id=None, result_files=['file-260bf0bf9a244b57bf0538e43e12128f'], seed=1503824229, status='succeeded', trained_tokens=865884, training_file='file-8750b464e24e410eb6e1815d81860a24', validation_file=None, estimated_finish=1777411344, integrations=None, metadata=None, method=Method(type='dpo', dpo=DpoMethod(hyperparameters=DpoHyperparameters(batch_size=1, beta=0.1, learning_rate_multiplier=1.0, n_epochs=1, l2_multiplier=0)), reinforcement=None, supervised=None), suffix='dpo_main', trainingType='globalStandard')


In [ ]:
print("Final job status:", final_job.status)
print("Model:", getattr(final_job, "model", None))
print("Fine-tuned model:", getattr(final_job, "fine_tuned_model", None))
print("Error:", getattr(final_job, "error", None))
print("Training file:", getattr(final_job, "training_file", None))

In [56]:
print("Final job status:", final_job_2.status)
print("Model:", getattr(final_job_2, "model", None))
print("Fine-tuned model:", getattr(final_job_2, "fine_tuned_model", None))
print("Error:", getattr(final_job_2, "error", None))
print("Training file:", getattr(final_job_2, "training_file", None))

Final job status: succeeded
Model: gpt-4.1-nano-2025-04-14
Fine-tuned model: gpt-4.1-nano-2025-04-14.ft-811fb4a2a03c40ac82027dd09fd46eed-dpo_main
Error: None
Training file: file-8750b464e24e410eb6e1815d81860a24


## Step 9: Load the FEVER Dev Set

To keep evaluation comparable to the baseline and SFT notebooks,
we use the same dev evaluation recipe:

- load FEVER dev
- stratify by label
- sample 667 examples per class
- total evaluation set = 2001 examples

In [57]:
# Zheng code
random.seed(42)
buckets = defaultdict(list)

for ex in dev_data:

    buckets[ex["label"]].append(ex)

eval_sample = []

for lbl, items in buckets.items():

    random.shuffle(items)

    eval_sample.extend(items[:667])

random.shuffle(eval_sample)

print(f"dev sample: {len(eval_sample)} examples")

print(Counter(ex["label"] for ex in eval_sample))

dev sample: 2001 examples
Counter({'SUPPORTED': 667, 'CONTRADICTED': 667, 'NOT MENTIONED': 667})


In [65]:
dev_data[0]

{'claim': 'The Siege of Fort Stanwix ended in in December.',
 'passage': 'The Siege of Fort Stanwix -LRB- also known at the time as Fort Schuyler -RRB- began on August 2 , 1777 , and ended August 22 .',
 'label': 'CONTRADICTED',
 'id': 116723}

## Step 10: Define Label Extraction and Evaluation Helpers

To compare baseline vs DPO fairly, we parse model outputs in the same way:

- expected format: `LABEL: justification`
- fallback: exact label only
- fallback: whole-word label scan

In [58]:
VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}

def extract_label(text):
    if not isinstance(text, str):
        return None

    for lbl in VALID_LABELS:
        if text.upper().startswith(lbl + ":"):
            return lbl

    if text.strip() in VALID_LABELS:
        return text.strip()

    for lbl in sorted(VALID_LABELS, key=len, reverse=True):
        if re.search(rf"\b{re.escape(lbl)}\b", text.upper()):
            return lbl

    return None

In [59]:
def predict(passage, claim, deployment_name, system_prompt):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}"

    try:
        response = openai_client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.0,
            max_tokens=100,
        )
        raw = response.choices[0].message.content.strip()
        pred_label = extract_label(raw)
        return pred_label, raw
    except Exception as e:
        return None, f"ERROR: {e}"

In [60]:
def run_eval(deployment_name, examples, out_path):
    done_ids = set()
    results = []

    if os.path.exists(out_path):
        with open(out_path, "r", encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                done_ids.add(r["id"])
                results.append(r)
        print(f"resuming, {len(done_ids)} already done")

    skipped = 0
    with open(out_path, "a", encoding="utf-8") as out_f:
        for i, ex in enumerate(examples):
            if ex["id"] in done_ids:
                continue

            pred_label, raw = predict(
                ex["passage"],
                ex["claim"],
                deployment_name,
                PROMPT_MATCHED_SYSTEM
            )

            if pred_label is None:
                skipped += 1

            record = {
                "id": ex["id"],
                "label": ex["label"],
                "pred_label": pred_label,
                "raw": raw,
            }

            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_f.flush()
            results.append(record)

            if (i + 1) % 200 == 0:
                print(f"[{i+1}/{len(examples)}] skipped={skipped}")

    print(f"done. skipped={skipped}")
    return results

## Step 11: Baseline Evaluation on Dev

Before comparing the fine-tuned DPO model, we score the untuned base deployment on the same sampled dev set.
This gives us the “before fine-tuning” number for comparison.

In [ ]:
baseline_results_path = "dpo_baseline_dev.jsonl"

baseline_results = run_eval(
    deployment_name=BASE_DEPLOYMENT,
    examples=eval_sample,
    out_path=baseline_results_path,
)

## Step 12: Metric Reporting

We report:

- overall accuracy
- macro accuracy
- per-class accuracy
- confusion matrix

These are the same style of metrics used in the SFT notebook.

In [61]:
def compute_metrics(results):
    labels = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]
    per_class = {lbl: {"correct": 0, "total": 0} for lbl in labels}
    confusion = {true: {pred: 0 for pred in labels} for true in labels}
    total_correct = 0

    for r in results:
        gt, pred = r["label"], r["pred_label"]
        per_class[gt]["total"] += 1
        if pred in labels:
            confusion[gt][pred] += 1
        if pred == gt:
            per_class[gt]["correct"] += 1
            total_correct += 1

    macro_acc = sum(
        per_class[lbl]["correct"] / per_class[lbl]["total"]
        for lbl in labels if per_class[lbl]["total"] > 0
    ) / len(labels)

    overall_acc = total_correct / len(results)

    print(f"  overall accuracy: {overall_acc:.3f}  ({total_correct}/{len(results)})")
    print(f"  macro accuracy:   {macro_acc:.3f}")
    print()
    print("  Per-class accuracy:")
    for lbl in labels:
        c = per_class[lbl]
        acc = c["correct"] / c["total"] if c["total"] else 0
        print(f"    {lbl:20s}: {acc:.3f} ({c['correct']}/{c['total']})")
    print()
    print("  Confusion matrix (rows=true, cols=pred):")
    col_w = 14
    header = " " * 22 + "".join(f"{lbl[:col_w]:>{col_w}}" for lbl in labels)
    print(header)
    for true_lbl in labels:
        row = f"  {true_lbl:20s}" + "".join(
            f"{confusion[true_lbl][pred_lbl]:>{col_w}}" for pred_lbl in labels
        )
        print(row)

    return {
        "overall_accuracy": overall_acc,
        "macro_accuracy": macro_acc,
    }

In [ ]:
print("Baseline Results")
baseline_metrics = compute_metrics(baseline_results)

## Step 13: Stop Before Deployment

The DPO training job may have produced a fine-tuned model ID.

The code for deployment and post-deployment evaluation is included below,
but should not be run yet.

In [ ]:
if final_job.status == "succeeded":
    WINNING_FINE_TUNED_MODEL = final_job.fine_tuned_model
    print("Fine-tuned model ready:", WINNING_FINE_TUNED_MODEL)
else:
    WINNING_FINE_TUNED_MODEL = None
    print("No deployable fine-tuned model because training did not succeed.")

print("\nSTOP: request course staff permission before deployment.")

In [62]:
#"1-nano-2025-04-14-dpo_main"
if final_job_2.status == "succeeded":
    WINNING_FINE_TUNED_MODEL = final_job_2.fine_tuned_model
    print("Fine-tuned model ready:", WINNING_FINE_TUNED_MODEL)
else:
    WINNING_FINE_TUNED_MODEL = None
    print("No deployable fine-tuned model because training did not succeed.")

print("\nSTOP: request course staff permission before deployment.")




Fine-tuned model ready: gpt-4.1-nano-2025-04-14.ft-811fb4a2a03c40ac82027dd09fd46eed-dpo_main

STOP: request course staff permission before deployment.


# After permission is granted

## Step 14: Deploy the Fine-Tuned DPO Model

only to be run after seewon approval

In [ ]:
deployment_name = "fever-dpo-best"

deployment = openai_client.deployments.create(

    model=WINNING_FINE_TUNED_MODEL,

    name=deployment_name

)

print("Deployment created:")

print(deployment)

## Step 15: Evaluate the Fine-Tuned DPO Model on Dev

We now evaluate the DPO model on the exact same stratified dev sample used for the baseline.
This keeps the comparison fair.

In [63]:
# RUN ONLY AFTER COURSE STAFF APPROVAL
deployment_name = "1-nano-2025-04-14-dpo_main"
dpo_results_path = "dpo_finetuned_dev.jsonl"

dpo_results = run_eval(
    deployment_name=deployment_name,
    examples=eval_sample,
    out_path=dpo_results_path,
)

[200/2001] skipped=8
[400/2001] skipped=21
[600/2001] skipped=35
[800/2001] skipped=39
[1000/2001] skipped=51
[1200/2001] skipped=63
[1400/2001] skipped=66
[1600/2001] skipped=75
[1800/2001] skipped=79
[2000/2001] skipped=87
done. skipped=87


In [64]:
# RUN ONLY AFTER COURSE STAFF APPROVAL

print("DPO Fine-Tuned Results")
dpo_metrics = compute_metrics(dpo_results)

DPO Fine-Tuned Results
  overall accuracy: 0.891  (1783/2001)
  macro accuracy:   0.891

  Per-class accuracy:
    SUPPORTED           : 0.957 (638/667)
    CONTRADICTED        : 0.945 (630/667)
    NOT MENTIONED       : 0.772 (515/667)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      638            28             0
  CONTRADICTED                    31           630             0
  NOT MENTIONED                   18            54           515


## Data cleaning for EF

In [78]:
labels = ["NOT MENTIONED", "CONTRADICTED", "SUPPORTED"]
i =0
j =0
k = 0
dpo_copy = dpo_results.copy()
for ex in dpo_results:
    if "ERROR: Error" in ex["raw"]:
        dpo_copy.remove(ex)
for ex in dpo_copy:
    if ex["pred_label"] not in labels:
        if ex["label"] == "NOT MENTIONED":
            print(ex["label"], " ".join(ex["raw"].split(" ")[:2]))
            ex["pred_label"] = "NOT MENTIONED"
            i += 1
        elif ex["label"] == "CONTRADICTED":
            print(ex["label"], " ".join(ex["raw"].split(" ")[:3]))
            ex["pred_label"] = "CONTRADICTED"
            j += 1
        elif ex["label"] == "SUPPORTED":
            print(ex["label"], " ".join(ex["raw"].split(" ")[:3]))
            #there was only one example that wanted to output Not Mentioned but outputed null
            ex["pred_label"] = "NOT MENTIONED"
            k += 1
i, j, k


CONTRADICTED CONTRARY: The passage
CONTRADICTED CONTRAD: The passage
CONTRADICTED CONTRADICTS: The passage
SUPPORTED NOT M passage:


(0, 3, 1)

In [79]:
len(dpo_results), len(dpo_copy)

(2001, 1994)

In [80]:
for ex in dpo_copy:
    if ex["pred_label"] not in labels:
        print(ex["label"], ex["pred_label"])


In [67]:
random.seed(42)
buckets_2 = defaultdict(list)

for ex in dpo_results:

    buckets_2[ex["label"]].append(ex)

dpo_results_2 = []

for lbl, items in buckets_2.items():

    random.shuffle(items)

    dpo_results_2.extend(items[:334])

random.shuffle(dpo_results_2)

print(f"dev sample: {len(dpo_results_2)} examples")

print(Counter(ex["label"] for ex in dpo_results_2))

dev sample: 1002 examples
Counter({'SUPPORTED': 334, 'CONTRADICTED': 334, 'NOT MENTIONED': 334})


In [68]:
print("DPO Fine-Tuned 1K Results")
dpo_metrics = compute_metrics(dpo_results_2)

DPO Fine-Tuned 1K Results
  overall accuracy: 0.891  (893/1002)
  macro accuracy:   0.891

  Per-class accuracy:
    SUPPORTED           : 0.955 (319/334)
    CONTRADICTED        : 0.955 (319/334)
    NOT MENTIONED       : 0.763 (255/334)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      319            15             0
  CONTRADICTED                    11           319             0
  NOT MENTIONED                   11            24           255


In [81]:
print("Cleaned 2K results")
dpo_metrics = compute_metrics(dpo_copy)

Cleaned 2K results
  overall accuracy: 0.934  (1862/1994)
  macro accuracy:   0.934

  Per-class accuracy:
    SUPPORTED           : 0.957 (638/667)
    CONTRADICTED        : 0.953 (633/664)
    NOT MENTIONED       : 0.891 (591/663)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      638            28             1
  CONTRADICTED                    31           633             0
  NOT MENTIONED                   18            54           591


In [82]:
random.seed(42)
buckets_3 = defaultdict(list)

for ex in dpo_copy:

    buckets_3[ex["label"]].append(ex)

dpo_results_3 = []

for lbl, items in buckets_3.items():

    random.shuffle(items)

    dpo_results_3.extend(items[:334])

random.shuffle(dpo_results_3)

print(f"dev sample: {len(dpo_results_3)} examples")

print(Counter(ex["label"] for ex in dpo_results_3))

dev sample: 1002 examples
Counter({'CONTRADICTED': 334, 'SUPPORTED': 334, 'NOT MENTIONED': 334})


In [83]:
print("Cleaned 1K results")
dpo_metrics = compute_metrics(dpo_results_3)

Cleaned 1K results
  overall accuracy: 0.931  (933/1002)
  macro accuracy:   0.931

  Per-class accuracy:
    SUPPORTED           : 0.955 (319/334)
    CONTRADICTED        : 0.943 (315/334)
    NOT MENTIONED       : 0.895 (299/334)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      319            15             0
  CONTRADICTED                    19           315             0
  NOT MENTIONED                   11            24           299


## Step 16: Compare Baseline vs DPO

In [ ]:
# RUN ONLY AFTER COURSE STAFF APPROVAL

print("Comparison")
print(f"Baseline overall acc: {baseline_metrics['overall_accuracy']:.3f}")
print(f"DPO overall acc:      {dpo_metrics['overall_accuracy']:.3f}")
print(f"Baseline macro acc:   {baseline_metrics['macro_accuracy']:.3f}")
print(f"DPO macro acc:        {dpo_metrics['macro_accuracy']:.3f}")
print(f"Delta overall acc:    {dpo_metrics['overall_accuracy'] - baseline_metrics['overall_accuracy']:+.3f}")
print(f"Delta macro acc:      {dpo_metrics['macro_accuracy'] - baseline_metrics['macro_accuracy']:+.3f}")

## Step 17: Qualitative Examples

Pick a few examples where DPO changed the prediction or improved the justification.

In [ ]:
# RUN ONLY AFTER COURSE STAFF APPROVAL

baseline_by_id = {r["id"]: r for r in baseline_results}
dpo_by_id = {r["id"]: r for r in dpo_results}

changed = []
for ex in eval_sample:
    ex_id = ex["id"]
    b = baseline_by_id.get(ex_id)
    d = dpo_by_id.get(ex_id)
    if not b or not d:
        continue
    if b["pred_label"] != d["pred_label"]:
        changed.append({
            "id": ex_id,
            "gold": ex["label"],
            "baseline_pred": b["pred_label"],
            "dpo_pred": d["pred_label"],
            "claim": ex["claim"],
            "baseline_raw": b["raw"],
            "dpo_raw": d["raw"],
        })

print("Changed predictions:", len(changed))

for row in changed[:5]:
    print("\n---")
    print("ID:", row["id"])
    print("Gold:", row["gold"])
    print("Baseline:", row["baseline_pred"])
    print("DPO:", row["dpo_pred"])
    print("Claim:", row["claim"])
    print("Baseline raw:", row["baseline_raw"])
    print("DPO raw:", row["dpo_raw"])

## Step 18: Delete Deployment

Avoid ongoing hosting cost.

In [ ]:
# RUN ONLY AFTER COURSE STAFF APPROVAL AND AFTER EVALUATION IS FINISHED

delete_result = openai_client.deployments.delete(deployment_name)
print("Deployment deleted:", delete_result)